# drfn-nllb2: controls for the NLLB memory result

- `gmm_free` (seeds 11, 22): the SAME memory module (8.0M params), routing NOT told the roots. If grounded gmm beats it, the gain is grounding, not extra capacity.
- `aux_w01` (seed 11): side heads with label-loss weight 0.1 instead of 0.5, to test whether the -2.6 chrF of side heads came from the label losses dominating training.


In [ ]:
!pip install -q sacrebleu sentencepiece
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.device_count())

In [ ]:
import glob, os, shutil, subprocess
def find(pat):
    hits = sorted(glob.glob('/kaggle/input/**/' + pat, recursive=True))
    if not hits: raise RuntimeError('not found under /kaggle/input: ' + pat)
    return hits[0]
pack_src = os.path.dirname(find('run_nllb2.sh'))
PACK = '/kaggle/working/pack'
if not os.path.exists(PACK): shutil.copytree(pack_src, PACK)
os.chdir(PACK)
from huggingface_hub import snapshot_download
print(snapshot_download('facebook/nllb-200-distilled-600M'))

In [ ]:
print('exit', subprocess.call(['bash', 'run_nllb2.sh']))

In [ ]:
import json
for p in sorted(glob.glob('/kaggle/working/nllb/*/eval.json')):
    r = json.load(open(p)); d = r['dev_beam5_nr4']; pr = r.get('probe') or {}; sw = r.get('slot_swap') or {}
    print('%-10s chrF %.2f chrF++ %.2f BLEU %.2f  best_ep %s  unseen own-slot %s  swap %s vs %s (n=%s)' % (
        r['run'], d['chrF_char'], d['chrF++'], d['BLEU'], r.get('best_epoch'),
        ('%.3f' % pr['own_slot_unseen']) if pr else '-',
        ('%.3f' % sw['success_swap']) if sw.get('n_items') else '-',
        ('%.3f' % sw['success_self']) if sw.get('n_items') else '-', sw.get('n_items', '-')))